In [ ]:
import torch
import pandas as pd
import numpy as np
import gc
import warnings
from tqdm import tqdm
from transformers import pipeline

warnings.filterwarnings("ignore")

In [ ]:
!pip install transformers torch pandas plotly openpyxl tqdm --quiet

In [ ]:
import torch
import pandas as pd
import numpy as np
import gc
import warnings
from tqdm import tqdm
from transformers import pipeline

warnings.filterwarnings("ignore")

In [ ]:
SAMPLE_SIZE = 2000
BATCH_SIZE = 8

In [ ]:
df = pd.read_excel("reviews.xlsx", engine="openpyxl")
df = df.dropna(subset=["comments"]).reset_index(drop=True)
df["comments"] = df["comments"].astype(str).str.strip()

df["sentiment"] = df["Label"].map({0: "negative", 1: "neutral", 2: "positive"}).fillna("neutral")

HIGH_TRIGGERS = ["сломался", "брак", "не работает", "опасно", "возврат", "мошенник", "подделка"]
MEDIUM_TRIGGERS = ["жалоба", "гарантия", "деньги", "обман", "разочарован", "не соответствует"]
TECH_TRIGGERS = ["глюк", "виснет", "греется", "медленно", "ошибка", "драйвер", "синий экран", "вылетает"]

def explain_criticality(row):
    reasons = []
    text = f"{str(row['comments'])} {str(row['disadvantages'])}".lower()

    # Тональность
    if row["sentiment"] == "negative":
        reasons.append("Тональность: негатив (+2)")
    elif row["sentiment"] == "neutral":
        reasons.append("Тональность: нейтраль/скрытый вопрос (+1)")

    # Критичные триггеры
    high_found = [w for w in HIGH_TRIGGERS if w in text]
    if high_found:
        reasons.append(f"Критичные маркеры: {', '.join(high_found)} (+{len(high_found)*3})")

    # Серьезные триггеры
    med_found = [w for w in MEDIUM_TRIGGERS if w in text]
    if med_found:
        reasons.append(f"Серьезные маркеры: {', '.join(med_found)} (+{len(med_found)*2})")

    # Технические триггеры
    tech_found = [w for w in TECH_TRIGGERS if w in text]
    if tech_found:
        reasons.append(f"Тех. проблемы: {', '.join(tech_found)} (+{len(tech_found)})")

    # Длина отзыва
    if len(str(row["comments"])) > 250:
        reasons.append("Подробное описание проблемы (+1)")

    # Вопрос
    comment_lower = str(row["comments"]).lower()
    has_question = "?" in comment_lower or any(q in comment_lower for q in ["как", "почему", "где", "когда"])
    if has_question and row["sentiment"] in ["negative", "neutral"]:
        reasons.append("Вопрос + негатив/нейтраль (+1)")

    score = row["criticality_score"]
    priority = row["priority_level"].upper()
    reason_str = " | ".join(reasons) if reasons else "Нет явных триггеров"
    return f"[{priority}] {score}/8: {reason_str}"

df["criticality_reason"] = df.apply(explain_criticality, axis=1)

priority_reviews = df[df["priority_level"].isin(["high", "medium"])].sort_values(
    "criticality_score", ascending=False
)

cols_to_show = [
    "product_name", "comments", "sentiment",
    "criticality_score", "priority_level", "criticality_reason"
]

high_count = len(df[df["priority_level"] == "high"])
med_count = len(df[df["priority_level"] == "medium"])


display(priority_reviews[cols_to_show].head(10).style.set_properties(**{
    'text-align': 'left',
    'max-width': '650px',
    'font-size': '12px'
}))

In [ ]:
device = 0 if torch.cuda.is_available() else -1

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

generator = pipeline(
    "text-generation",
    model=model_name,
    device=device,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    max_new_tokens=150,
    do_sample=False,
    temperature=0.3,
    truncation=True,
    return_full_text=False
)

In [ ]:
def build_prompt(row):
    sent_map = {"negative": "негативный", "neutral": "нейтральный/с вопросом", "positive": "позитивный"}
    return f"""Ты — специалист службы поддержки интернет-магазина. ОТВЕЧАЙ ИСКЛЮЧИТЕЛЬНО НА РУССКОМ ЯЗЫКЕ. Запрещено использовать английские слова, фразы или приветствия. Напиши краткий, человечный и вежливый ответ на отзыв.
Правила:
1. Все ответы должны быть исключительно на русском языке. Не используй английские слова и фразы никакие.
2. Начни с благодарности за отзыв и покупку.
3. Тон: теплый, эмпатичный, без канцеляризмов и шаблонных фраз.
4. Если отзыв позитивный: поблагодари, отметь упомянутые плюсы.
5. Если есть простой вопрос или замечание по настройке/инструкции: кратко ответь и предложи посмотреть раздел [Ссылка на FAQ или Инструкцию].
6. Если проблема сложная, товар сломался или требует ремонта/возврата: вырази сожаление, предложи конкретное действие и укажи контакт: [Телефон поддержки] или [Ссылка на обращение в сервис].
7. Не используй эмодзи. Длина: 1-3 предложения.

Данные:
- Тон: {sent_map[row['sentiment']]}
- Отзыв: {str(row['comments'])[:300]}
- Плюсы: {str(row['advantages'])[:100]}
- Минусы: {str(row['disadvantages'])[:100]}

Ответ:"""

print("Запуск пакетной генерации ответов...")
prompts = [build_prompt(row) for _, row in df.iterrows()]
df["ai_reply"] = ""

# Пакетная обработка (ускоряет работу в 4-6 раз)
for start in tqdm(range(0, len(prompts), BATCH_SIZE), desc="Обработка пакетов"):
    end = min(start + BATCH_SIZE, len(prompts))
    batch = prompts[start:end]

    try:
        results = generator(batch, batch_size=BATCH_SIZE, max_new_tokens=120)
        replies = [r[0]["generated_text"].strip().replace("\n", " ") for r in results]
    except Exception as e:
        print(f"Ошибка в пакете {start}-{end}: {e}")
        replies = ["Спасибо за ваш отзыв. Наши специалисты свяжутся с вами для уточнения деталей."] * len(batch)

    for i, idx in enumerate(df.index[start:end]):
        df.at[idx, "ai_reply"] = replies[i]

print("Генерация завершена.")

Запуск пакетной генерации ответов...


Обработка пакетов:   0%|          | 0/250 [00:00<?, ?it/s]A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Обработка пакетов:   0%|          | 1/250 [02:57<12:17:50, 177.79s/it]A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Обработка пакетов:   1%|    

Генерация завершена.


In [ ]:
!pip install plotly wordcloud matplotlib --quiet

import plotly.graph_objects as go
import plotly.express as px
from wordcloud import WordCloud
import matplotlib.pyplot as plt
from IPython.display import display, HTML
import pandas as pd

# Настройки отображения
pd.set_option('display.max_colwidth', 200)
plt.rcParams['font.size'] = 10

In [ ]:
fig_table = go.Figure(data=[go.Table(
    header=dict(
        values=["Товар", "Отзыв", "Тональность", "Приоритет", "Балл", "Ответ"],
        fill_color='#2c3e50',
        font=dict(color='white', size=11),
        align='left',
        height=30
    ),
    cells=dict(
        values=[
            df["product_name"].str[:35] + "...",
            df["comments"].str[:90] + "...",
            df["sentiment"].map({"negative": "Негатив", "neutral": "Нейтраль", "positive": "Позитив"}),
            df["priority_level"].map({"high": "Высокий", "medium": "Средний", "standard": "Стандарт"}),
            df["criticality_score"],
            df["ai_reply"].fillna("").str[:70] + "..."
        ],
        align='left',
        font=dict(size=10),
        height=25
    )
)])

fig_table.update_layout(
    title="Обзор отзывов с ответами и приоритетами",
    height=450,
    margin=dict(l=10, r=10, t=50, b=10),
    template="plotly_white"
)
fig_table.show()

In [ ]:
priority_sent = df.groupby(["priority_level", "sentiment"]).size().unstack(fill_value=0)
priority_order = ["high", "medium", "standard"]
priority_labels = {"high": "Высокий", "medium": "Средний", "standard": "Стандарт"}

fig1 = go.Figure(data=[
    go.Bar(name="Позитив", x=[priority_labels[p] for p in priority_order],
           y=[priority_sent.get("positive", [0]*3)[i] for i in range(3)], marker_color="#27ae60"),
    go.Bar(name="Нейтраль", x=[priority_labels[p] for p in priority_order],
           y=[priority_sent.get("neutral", [0]*3)[i] for i in range(3)], marker_color="#f39c12"),
    go.Bar(name="Негатив", x=[priority_labels[p] for p in priority_order],
           y=[priority_sent.get("negative", [0]*3)[i] for i in range(3)], marker_color="#c0392b")
])
fig1.update_layout(
    barmode="group",
    title="Распределение отзывов по приоритету и тональности",
    xaxis_title="Уровень приоритета",
    yaxis_title="Количество отзывов",
    height=350,
    template="plotly_white",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)
fig1.show()

cat_sent = df.groupby(["category", "sentiment"]).size().unstack(fill_value=0)
fig2 = go.Figure(data=[
    go.Bar(name="Позитив", x=cat_sent.index, y=cat_sent.get("positive", [0]*len(cat_sent)), marker_color="#27ae60"),
    go.Bar(name="Нейтраль", x=cat_sent.index, y=cat_sent.get("neutral", [0]*len(cat_sent)), marker_color="#f39c12"),
    go.Bar(name="Негатив", x=cat_sent.index, y=cat_sent.get("negative", [0]*len(cat_sent)), marker_color="#c0392b")
])
fig2.update_layout(
    barmode="group",
    title="Тональность отзывов по категориям товаров",
    xaxis_tickangle=-45,
    height=350,
    template="plotly_white",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)
fig2.show()


high_priority = df[df["priority_level"] == "high"]
top_products = high_priority.groupby("product_name").size().nlargest(12)

if len(top_products) > 0:
    fig3 = go.Figure(data=go.Bar(
        x=top_products.index.str[:30] + "...",
        y=top_products.values,
        marker_color="#c0392b",
        text=top_products.values,
        textposition="outside"
    ))
    fig3.update_layout(
        title="Товары с наибольшим количеством обращений высокого приоритета",
        xaxis_tickangle=-45,
        height=400,
        xaxis_title="Товар",
        yaxis_title="Обращений (высокий приоритет)",
        template="plotly_white"
    )
    fig3.show()
else:
    print("Обращений высокого приоритета не обнаружено.")

In [ ]:
# примеры
priority_sample = df[df["priority_level"].isin(["high", "medium"])].head(8)

if len(priority_sample) > 0:

    for idx, row in priority_sample.iterrows():
        priority_label = row["priority_level"].upper()
        sent_label = row["sentiment"].capitalize()

        print(f"\nТовар: {row['product_name'][:50]}")
        print(f"Приоритет: {priority_label} | Тональность: {sent_label} | Балл: {row['criticality_score']}/8")
        print(f"Отзыв: {str(row['comments'])[:150]}...")
        print(f"Ответ: {row['ai_reply']}")
        print("-" * 80)
else:
    print("Приоритетные обращения не найдены в текущей выборке.")